<a href="https://colab.research.google.com/github/CodeWithGB/my_minor_projects/blob/main/GroupDNA_Gaurav_DS24039.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GroupDNA: WhatsApp Chat Analyzer
**Name:** Gaurav Mahesh Bodhe
**Roll Number:** DS24039
**Batch:** September
**Date:** 02/10/2026

In [4]:
import numpy as np
from datetime import datetime, timedelta

# ==========================================
# FEATURE 1: THE CHAT PARSER
# ==========================================
messages = []
system_msgs = 0
media_msgs = 0
deleted_msgs = 0
participants = set()

with open('hostel_bois.txt', 'r', encoding='utf-8') as f:
    lines = f.readlines()

for line in lines:
    line = line.strip()
    if not line:
        continue

    # Split timestamp from the rest of the message
    # Using maxsplit=1 so we don't accidentally split the message text
    parts = line.split(' - ', 1)
    if len(parts) != 2:
        system_msgs += 1
        continue

    timestamp_str, rest = parts

    # System messages usually don't have a colon for the sender
    if ': ' not in rest:
        system_msgs += 1
        continue

    sender, text = rest.split(': ', 1)
    participants.add(sender)

    is_media = (text == '<Media omitted>')
    is_deleted = (text == 'This message was deleted')

    if is_media:
        media_msgs += 1
    elif is_deleted:
        deleted_msgs += 1

    try:
        dt = datetime.strptime(timestamp_str, '%d/%m/%y, %H:%M')
    except ValueError:
        system_msgs += 1
        continue
    # AI-assisted: I used an AI assistant to help ensure my NumPy matrix and datetime parsing adhered to the project constraints.

    messages.append({
        'dt': dt,
        'timestamp_str': timestamp_str,
        'sender': sender,
        'text': text,
        'is_media': is_media,
        'is_deleted': is_deleted,
        'is_real_text': not (is_media or is_deleted)
    })

# ==========================================
# FEATURE 2: GROUP OVERVIEW
# ==========================================
total_messages = len(messages)
start_date = messages[0]['dt']
end_date = messages[-1]['dt']
duration_days = (end_date - start_date).days + 1

msg_counts = {}
for m in messages:
    sender = m['sender']
    msg_counts[sender] = msg_counts.get(sender, 0) + 1

# Sort participants by message count (highest to lowest)
participant_list = sorted(list(participants), key=lambda x: msg_counts[x], reverse=True)

# ==========================================
# FEATURE 3: MOST ACTIVE DAY AND HOUR
# ==========================================
day_counts = {}
hour_counts = {}

for m in messages:
    day_str = m['dt'].strftime('%d %B %Y')
    hour = m['dt'].hour
    day_counts[day_str] = day_counts.get(day_str, 0) + 1
    hour_counts[hour] = hour_counts.get(hour, 0) + 1

busiest_day = max(day_counts, key=day_counts.get)
busiest_day_count = day_counts[busiest_day]
busiest_hour = max(hour_counts, key=hour_counts.get)

# ==========================================
# FEATURE 4: ACTIVITY HEATMAP (NumPy)
# ==========================================
# 6 participants (rows) x 24 hours (columns)
heatmap_matrix = np.zeros((len(participant_list), 24), dtype=int)

for m in messages:
    p_idx = participant_list.index(m['sender'])
    h_idx = m['dt'].hour
    heatmap_matrix[p_idx, h_idx] += 1

def get_block(val, max_val):
    """Helper function to render heatmap blocks based on percentage."""
    if max_val == 0 or val == 0: return "."
    ratio = val / max_val
    if ratio <= 0.25: return "."
    elif ratio <= 0.50: return "░"
    elif ratio <= 0.75: return "▒"
    else: return "█"

# ==========================================
# FEATURE 5: TOP WORDS
# ==========================================
stop_words = {'i', 'is', 'the', 'a', 'and', 'or', 'to', 'of', 'in', 'on', 'for', 'it', 'this', 'that', 'with', 'you', 'my', 'me', 'so', 'was', 'but', 'have', 'just', 'not', 'be', 'are', 'we', 'they', 'all', 'like', 'what', 'how', 'why', 'who', 'when', 'then', 'if', 'about', 'from', 'as', 'at', 'by', 'an', 'hai', 'ki', 'se', 'ko', 'tu', 'hu', 'ho', 'tha'}

word_counts = {}
for m in messages:
    if not m['is_real_text']:
        continue

    text = m['text'].lower()

    # Strip basic punctuation without using regex
    for char in '.,?!;:"()[]{}-_*\n\t~':
        text = text.replace(char, ' ')

    words = text.split()
    for w in words:
        if w not in stop_words and len(w) > 1:
            word_counts[w] = word_counts.get(w, 0) + 1

top_words = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)[:5]
max_word_count = top_words[0][1] if top_words else 1

# ==========================================
# FEATURE 6: RESPONSE SPEED & SILENT STREAKS
# ==========================================
response_times = {p: [] for p in participant_list}
last_sender = None
last_time = None

for m in messages:
    sender = m['sender']
    dt = m['dt']
    # If someone replies to someone else, log the time gap
    if last_sender and last_sender != sender:
        gap = (dt - last_time).total_seconds()
        response_times[sender].append(gap)
    last_sender = sender
    last_time = dt

avg_response = {}
for p, gaps in response_times.items():
    avg_response[p] = sum(gaps) / len(gaps) if gaps else 0

# Calculate silent streaks
all_dates = [start_date.date() + timedelta(days=i) for i in range(duration_days)]
longest_streak = {p: 0 for p in participant_list}
streak_dates = {p: "" for p in participant_list}

for p in participant_list:
    p_dates = {m['dt'].date() for m in messages if m['sender'] == p}
    current_streak = 0
    max_streak = 0
    streak_start = None
    best_streak_start = None
    best_streak_end = None

    for d in all_dates:
        if d not in p_dates:
            if current_streak == 0:
                streak_start = d
            current_streak += 1
            if current_streak > max_streak:
                max_streak = current_streak
                best_streak_start = streak_start
                best_streak_end = d
        else:
            current_streak = 0

    longest_streak[p] = max_streak
    if max_streak > 0:
        # Format the date range nicely for the report
        streak_dates[p] = f"({best_streak_start.strftime('%d %b')} to {best_streak_end.strftime('%d %b')})"

# ==========================================
# FEATURE 7: PERSONALITY ARCHETYPES
# ==========================================
raw_scores = {p: {} for p in participant_list}

for p in participant_list:
    p_msgs = [m for m in messages if m['sender'] == p]
    p_real_msgs = [m for m in p_msgs if m['is_real_text']]

    # 1. The Spammer (Consecutive burst tracking)
    bursts, current_burst, prev_sender = [], 0, None
    for m in messages:
        if m['sender'] == p:
            if prev_sender == p or prev_sender is None:
                current_burst += 1
            else:
                current_burst = 1
        else:
            if prev_sender == p and current_burst > 0:
                bursts.append(current_burst)
                current_burst = 0
        prev_sender = m['sender']
    if prev_sender == p and current_burst > 0:
        bursts.append(current_burst)

    avg_burst = sum(bursts) / len(bursts) if bursts else 0

    # 2. The Group Mom
    caring_words = ['okay', 'safe', 'eat', 'sleep', 'take care', 'are you', 'please', 'reminder', 'drink water', "don't forget"]
    mom_score = sum(1 for m in p_real_msgs if any(cw in m['text'].lower() for cw in caring_words))

    # 3. The Night Owl
    night_msgs = sum(1 for m in p_msgs if m['dt'].hour >= 23 or m['dt'].hour <= 4)
    night_ratio = (night_msgs / len(p_msgs)) * 100 if p_msgs else 0

    # 4. The Storyteller
    words_per_msg = [len(m['text'].split()) for m in p_real_msgs]
    avg_words = sum(words_per_msg) / len(words_per_msg) if words_per_msg else 0

    # 5. The Drama Queen
    drama_msgs = sum(1 for m in p_real_msgs if (m['text'].isupper() and len(m['text']) >= 3) or m['text'].count('!') >= 2)
    drama_ratio = (drama_msgs / len(p_real_msgs)) * 100 if p_real_msgs else 0

    # 6. The Ghost
    p_dates = {m['dt'].date() for m in p_msgs}
    silent_days = duration_days - len(p_dates)
    ghost_ratio = (silent_days / duration_days) * 100

    # BONUS ARCHETYPE: The Late-Night Philosopher
    deep_words = ['life', 'universe', 'meaning', 'point', 'exist', 'matter', 'thoughts', 'dread']
    philosopher_msgs = sum(1 for m in p_real_msgs if (m['dt'].hour >= 23 or m['dt'].hour <= 4) and any(dw in m['text'].lower() for dw in deep_words))

    # Normalize scores so we can compare them (score / threshold)
    raw_scores[p] = {
        'THE SPAMMER': (avg_burst / 3.0, f"(avg {avg_burst:.1f} msgs in a row)"),
        'THE GROUP MOM': (mom_score / 10.0, f"(caring keyword score: {mom_score})"),
        'THE NIGHT OWL': (night_ratio / 60.0, f"({night_ratio:.1f}% msgs between 23h-04h)"),
        'THE STORYTELLER': (avg_words / 30.0, f"(avg {avg_words:.1f} words per msg)"),
        'THE DRAMA QUEEN': (drama_ratio / 30.0, f"({drama_ratio:.1f}% ALL-CAPS messages)"),
        'THE GHOST': (ghost_ratio / 60.0, f"(silent on {silent_days} of {duration_days} days)"),
        'THE PHILOSOPHER': (philosopher_msgs / 5.0, f"(used deep words at night {philosopher_msgs} times)") # Bonus
    }

# Distribute archetypes exclusively
all_scores = []
for p in participant_list:
    for arch, (score, reason) in raw_scores[p].items():
        all_scores.append((score, p, arch, reason))

# Sort by highest relative score first
all_scores.sort(key=lambda x: x[0], reverse=True)

archetypes = {}
assigned_archetypes = set()
for score, p, arch, reason in all_scores:
    if p not in archetypes and arch not in assigned_archetypes:
        archetypes[p] = (arch, reason)
        assigned_archetypes.add(arch)

# ==========================================
# FEATURE 8: THE FINAL REPORT (Visual Layout)
# ==========================================
print("="*60)
print(f"{'GROUPDNA REPORT':^60}")
print("="*60)
print(f"Group          : \"Hostel Bois 4ever\"")
print(f"Total messages : {total_messages:,} messages")
print(f"Participants   : {len(participant_list)} members")
print("="*60)
print(f"Period         : {start_date.strftime('%d %B %Y')} to {end_date.strftime('%d %B %Y')} ({duration_days} days)")
print(f"Busiest day    : {busiest_day} ({busiest_day_count} messages)")
print(f"Busiest hour   : {busiest_hour:02d}:00 - {(busiest_hour+1)%24:02d}:00")
print()

print("MESSAGES PER PERSON")
for p in participant_list:
    pct = (msg_counts[p] / total_messages) * 100
    print(f"{p:<10} {msg_counts[p]:>5} ({pct:.1f}%)")
print()

print("ACTIVITY HEATMAP (messages by hour, 3-hour blocks)")
print(f"{'':<8} 00 03 06 09 12 15 18 21")
for i, p in enumerate(participant_list):
    # Binning the 24 hours into 8 blocks of 3 hours to make it look clean like the brief
    chunks = [sum(heatmap_matrix[i, h:h+3]) for h in range(0, 24, 3)]
    row_max = max(chunks) if max(chunks) > 0 else 1
    blocks = [get_block(val, row_max) for val in chunks]
    print(f"{p:<8} " + "  ".join(blocks))
print()

print("THIS GROUP'S FAVOURITE WORDS")
for w, c in top_words:
    # Creates the horizontal bar chart
    bar = "█" * int((c / max_word_count) * 20)
    print(f"{w:<10} {c:<5} {bar}")
print()

print("RESPONSE PATTERNS")
fastest = min(avg_response, key=avg_response.get)
slowest = max(avg_response, key=avg_response.get)
print(f"Fastest replier: {fastest} (avg {avg_response[fastest]/60:.1f} minutes)")
print(f"Slowest replier: {slowest} (avg {avg_response[slowest]/3600:.1f} hours)")
print()

print("LONGEST SILENT STREAKS")
streak_ranking = sorted(participant_list, key=lambda x: longest_streak[x], reverse=True)
for p in streak_ranking[:3]:
    print(f"{p:<10} : {longest_streak[p]} days {streak_dates[p]}")
print()

print("PERSONALITY ARCHETYPES")
for p in participant_list:
    # Fallback just in case someone didn't trigger any archetype properly
    arch, reason = archetypes.get(p, ("THE OBSERVER", "(Just hanging around)"))
    print(f"{p:<10} -> {arch:<18} {reason}")
print()
print("="*60)
print(f"{'Generated by GroupDNA':^60}")
print(f"{'Built with Python + NumPy':^60}")
print("="*60)

                      GROUPDNA REPORT                       
Group          : "Hostel Bois 4ever"
Total messages : 3,174 messages
Participants   : 6 members
Period         : 01 April 2024 to 30 May 2024 (60 days)
Busiest day    : 04 May 2024 (76 messages)
Busiest hour   : 18:00 - 19:00

MESSAGES PER PERSON
Rahul        953 (30.0%)
Priya        718 (22.6%)
Neha         635 (20.0%)
Aman         490 (15.4%)
Karan        354 (11.2%)
Vikas         24 (0.8%)

ACTIVITY HEATMAP (messages by hour, 3-hour blocks)
         00 03 06 09 12 15 18 21
Rahul    .  .  ░  ░  ▒  █  █  █
Priya    .  .  ░  █  █  ▒  ▒  ░
Neha     .  .  ░  █  ▒  ▒  █  ▒
Aman     █  █  .  .  .  .  .  ░
Karan    .  .  .  ▒  █  █  █  ░
Vikas    .  .  █  ░  █  █  █  █

THIS GROUP'S FAVOURITE WORDS
guys       318   ████████████████████
am         260   ████████████████
today      257   ████████████████
he         220   █████████████
his        217   █████████████

RESPONSE PATTERNS
Fastest replier: Rahul (avg 34.9 minutes)
Slowest